# 02 — Classificação: o voo vai atrasar?

**Tarefa:** classificação binária `IS_DELAYED = 1` se o voo atrasar ≥ 15 min na chegada.

**Algoritmos comparados (≥2):**
1. **Regressão Logística** (baseline linear, com `class_weight='balanced'`).
2. **LightGBM** (gradient boosting em árvores).

**Métricas:** Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC.

**Cuidados antivazamento:** removemos do X todas as colunas de `config.LEAKY_COLS` — variáveis só conhecidas *após* o voo (e.g. `DEPARTURE_DELAY`, `TAXI_OUT`, `*_DELAY` granulares).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from src import config
from src.data_loader import load_flights
from src.preprocessing import clean_pipeline
from src.features import feature_pipeline
from src.models.classification import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    build_logistic_pipeline,
    build_lightgbm_pipeline,
)
from src.models.regression import build_feature_matrix
from src.evaluation import (
    classification_metrics,
    plot_confusion_matrix,
    plot_roc_curves,
    print_classification_report,
    metrics_table,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Preparar dados

In [ ]:
df = load_flights(sample_size=config.SAMPLE_SIZE)
df = clean_pipeline(df)
df = feature_pipeline(df)
print(f"Shape pós-limpeza: {df.shape}")
print(f"Taxa de atraso: {df['IS_DELAYED'].mean():.2%}")

In [ ]:
X = build_feature_matrix(df)
y = df["IS_DELAYED"].values

print("Features numéricas:", NUMERIC_FEATURES)
print("Features categóricas:", CATEGORICAL_FEATURES)
print(f"\nX.shape={X.shape}, y mean={y.mean():.4f}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=config.RANDOM_SEED, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Modelo 1 — Regressão Logística

Baseline linear. `class_weight='balanced'` compensa o desbalanceamento das classes.

In [ ]:
import time

lr = build_logistic_pipeline(config.RANDOM_SEED)
t0 = time.time()
lr.fit(X_train, y_train)
print(f"Treinamento: {time.time() - t0:.1f}s")

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]
metrics_lr = classification_metrics(y_test, y_pred_lr, y_proba_lr)
metrics_lr

In [ ]:
print_classification_report(y_test, y_pred_lr, "Regressão Logística")

## 3. Modelo 2 — LightGBM

In [ ]:
lgbm = build_lightgbm_pipeline(config.RANDOM_SEED, n_estimators=500, learning_rate=0.05)
t0 = time.time()
lgbm.fit(X_train, y_train)
print(f"Treinamento: {time.time() - t0:.1f}s")

y_pred_lgbm = lgbm.predict(X_test)
y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]
metrics_lgbm = classification_metrics(y_test, y_pred_lgbm, y_proba_lgbm)
metrics_lgbm

In [ ]:
print_classification_report(y_test, y_pred_lgbm, "LightGBM")

## 4. Comparação

In [ ]:
summary = metrics_table({
    "Logistic Regression": metrics_lr,
    "LightGBM": metrics_lgbm,
})
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_confusion_matrix(y_test, y_pred_lr, ax=axes[0], title="Logistic Regression")
plot_confusion_matrix(y_test, y_pred_lgbm, ax=axes[1], title="LightGBM")
plot_roc_curves(
    {
        "Logistic": (y_test, y_proba_lr),
        "LightGBM": (y_test, y_proba_lgbm),
    },
    ax=axes[2],
)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "08_classification_results.png", dpi=120)
plt.show()

## 5. Importância das features (LightGBM)

In [ ]:
preproc = lgbm.named_steps["preprocess"]
clf = lgbm.named_steps["clf"]
feature_names = preproc.get_feature_names_out()
imp = (
    pd.Series(clf.feature_importances_, index=feature_names)
    .sort_values(ascending=False)
    .head(20)
)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=imp.values, y=imp.index, hue=imp.index, palette="viridis", ax=ax, legend=False)
ax.set_title("Top 20 features (LightGBM, ganho)")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "09_feature_importance_clf.png", dpi=120)
plt.show()

## 6. Tuning do threshold (precision × recall)

Em operações reais (compensação a passageiro / alocação de equipe) o custo
de falsos positivos e falsos negativos é assimétrico. Ajustar o threshold
permite navegar essa curva.

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_proba_lgbm)
f1_curve = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = int(np.nanargmax(f1_curve))
print(f"Threshold ótimo por F1: {thresholds[best_idx]:.3f}")
print(f"  precision={precision[best_idx]:.3f}, recall={recall[best_idx]:.3f}, f1={f1_curve[best_idx]:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, precision[:-1], label="Precision")
ax.plot(thresholds, recall[:-1], label="Recall")
ax.plot(thresholds, f1_curve[:-1], label="F1", linestyle="--")
ax.axvline(thresholds[best_idx], color="red", alpha=0.5)
ax.set_xlabel("Threshold")
ax.set_title("Precision/Recall/F1 vs threshold (LightGBM)")
ax.legend()
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "10_threshold_tuning.png", dpi=120)
plt.show()

## 7. Persistir o melhor modelo

In [ ]:
best_model = lgbm
out_path = config.MODELS_DIR / "clf_lightgbm_best.joblib"
joblib.dump(best_model, out_path)
print(f"Modelo salvo em {out_path}")
summary.to_csv(config.REPORTS_DIR / "metrics_classification.csv")

## 8. Discussão

**Conclusões esperadas** (a confirmar após execução):
- LightGBM supera a regressão logística em todas as métricas — especialmente ROC-AUC e PR-AUC.
- Sem `DEPARTURE_DELAY` no X, prever atraso *antes* do voo decolar é difícil: o ROC-AUC dificilmente passa de ~0.70-0.72 com as features disponíveis.
- As features mais informativas tendem a ser: hora de partida agendada, companhia, aeroporto de origem, mês, duração agendada.

**Limitações**:
- Não temos dados meteorológicos *previstos* — apenas o atraso meteorológico observado (que é leaky). Incorporar previsões reais melhoraria muito o modelo.
- O dataset é só de 2015 — não captura mudanças estruturais da indústria (pós-COVID, p.ex.).
- Para uso em produção, seria necessário avaliar drift sazonal e re-treino periódico.

**Próximos passos**:
- Adicionar dados de clima histórico por aeroporto/data.
- Engenharia de features de "propagação": atraso médio do aeroporto na última hora.
- Calibração de probabilidades (Platt / isotônica) para uso em decisões de negócio.